# Preflight: verify the environment before class

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cybertide-solutions/ai-platform-architect-labs/blob/main/notebooks/00_preflight.ipynb)

**Start:** run the first code cell. It automatically downloads the supporting files in a fresh Colab runtime. No ZIP upload, local installation or pasted setup code is required.

Run this 3–5 working days before delivery. This checks exact endpoint capabilities. A successful chat call alone does not prove tools or embeddings work.

In [ ]:
# Automatic lab setup: run this cell first. No ZIP upload or manual clone is needed.
from pathlib import Path
import os, sys, json, tempfile, subprocess, shutil

def prepare_lab_files(colab_home=Path('/content')):
    required = ('engine/__init__.py', 'engine/knowledge.py', 'data/policies.json', 'data/orders.json')
    def ready(path):
        return all((path / name).is_file() for name in required)
    try:
        import google.colab
        in_colab = True
    except ImportError:
        in_colab = False
    target = colab_home / 'ai-platform-architect-labs'
    candidates = ([target] if in_colab else []) + [Path.cwd(), Path.cwd().parent]
    root = next((path for path in candidates if ready(path)), None)
    if root is None and in_colab:
        if target.exists():
            raise RuntimeError('An incomplete course folder already exists. Save your work, then disconnect and delete this Colab runtime and reopen the notebook.')
        print('Downloading the course code and data from GitHub...')
        colab_home.mkdir(parents=True, exist_ok=True)
        staging = Path(tempfile.mkdtemp(prefix='course-download-', dir=colab_home))
        try:
            result = subprocess.run(
                ['git', 'clone', '--quiet', '--depth', '1', '--branch', 'main',
                 'https://github.com/cybertide-solutions/ai-platform-architect-labs.git', str(staging / 'repo')],
                capture_output=True, text=True, timeout=120)
            if result.returncode != 0 or not ready(staging / 'repo'):
                raise RuntimeError('Course download failed. Check the runtime internet connection and rerun this cell. Your existing work was not changed.')
            (staging / 'repo').rename(target)
            root = target
        except subprocess.TimeoutExpired:
            raise RuntimeError('Course download timed out. Rerun this cell to try again.') from None
        finally:
            shutil.rmtree(staging, ignore_errors=True)
    if root is None:
        raise RuntimeError('Local Jupyter: open this notebook from the extracted course folder. In Colab, use the README Open in Colab button.')
    root = root.resolve()
    os.chdir(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print('Lab files ready. Continue with the cells below.')
    return root

ROOT = prepare_lab_files()
# End automatic setup
from engine import read_data
from engine.identity import BUYER, FINANCE, BEACON
from engine.knowledge import Index, evidence_errors
from engine.purchasing import Purchases, query_errors
from engine.platform import Platform
from engine.provider import ModelClient
# Deliberately opt in. No credential is required for software rehearsal.
# Colab only: enable after saving these names in Colab Secrets and granting access.
USE_COLAB_SECRETS=False
if USE_COLAB_SECRETS:
    from google.colab import userdata
    for name in ['CHAT_URL','CHAT_MODEL','CHAT_KEY','EMBED_URL','EMBED_MODEL','EMBED_KEY']:
        os.environ[name]=userdata.get(name)
    os.environ['LIVE_MODE']='1'
LIVE=os.environ.get('LIVE_MODE')=='1'
client=ModelClient.environment() if LIVE else None
print('LIVE MODEL' if LIVE else 'SOFTWARE REHEARSAL: no generated AI answers or AI quality score')

**Local route:** Python 3.11+ and Jupyter. Core code uses only the standard library. **Colab route:** click the README launch button and run the first code cell. The notebook downloads its supporting code and data automatically. No ZIP upload or manual clone is needed.

Set secrets through the private prompt cell below. Only use fictional supplied data. The client expects an HTTPS base URL exposing `/chat/completions` and, for the data-plane extension, `/embeddings`. Select exact models with the trainer; provider APIs differ. Do not print keys or save them in notebooks.

In [ ]:
import getpass
# Change to True only after the trainer confirms account, endpoint and budget.
CONFIGURE=False
if CONFIGURE:
    for prefix in ['CHAT','EMBED']:
        os.environ[prefix+'_URL']=input(prefix+' HTTPS base URL: ').strip()
        os.environ[prefix+'_MODEL']=input(prefix+' exact model ID: ').strip()
        os.environ[prefix+'_KEY']=getpass.getpass(prefix+' key (hidden): ')
    os.environ['LIVE_MODE']='1'
    client=ModelClient.environment(); LIVE=True

In [ ]:
ix=Index(); ix.build('v1',read_data('policies.json'))
purchases=Purchases()
assert purchases.summary(FINANCE,{'quarter':'2026-Q1','supplier':'Nova'})['total_minor']==6500000
assert ix.search('standard supplier payment terms',BUYER)[0]['id']=='A-TERMS'
print('Python, data, SQL and retrieval ready')
if LIVE:
    from engine.purchasing import QUERY_PROMPT, TOOLS
    result=client.json(QUERY_PROMPT,'Nova spend in 2026-Q1?')
    assert not query_errors(result), result
    message=client.message([{'role':'user','content':'Use spend_summary to find Nova spend in 2026-Q1.'}],tools=TOOLS)
    assert message.get('tool_calls'), 'Tool selection not observed; investigate this model/prompt.'
    embed=ModelClient.environment('EMBED')
    vectors=embed.vectors(['Supplier payment policy.','Nova spend summary.'])
    assert len(vectors)==2
    print('JSON, native tools and real embedding endpoint observed; dimensions:',len(vectors[0]))
    print('Request telemetry (no secrets):',client.calls)
else:
    print('LIVE READINESS PENDING. Trainer must execute JSON, tool and embedding checks.')
ix.close(); purchases.close()

**Record:** date, exact model IDs, endpoint provider, observed JSON/tool/embedding outcomes, request allowance, account owner and fallback owner. If `response_format` or tool schema is unsupported, select a supported model or adapt and retest the adapter before class. Do not silently remove a contract.

**Gate:** all teams can run software rehearsal; at least one approved live account per team can run the three capability checks. If live readiness fails, reschedule the model experiment or use clearly labelled trainer-captured live evidence. Offline fixtures cannot satisfy the AI quality assessment.